<a href="https://colab.research.google.com/github/Of-Calls/sisicallcall-verification-finetuning/blob/main/sisicallcall_titanet_small_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TitaNet-Small Fine-tuning Sweep Notebook

## 목적
이전 실험에서 누락됐던 변수들을 **CONFIG 딕셔너리 하나**로 통제해서
하이퍼파라미터/loss/recipe를 다양하게 조합 비교하기 위한 노트북.

## 한 줄만 바꾸면 시나리오 전환
```python
CONFIG = PRESETS["bn_affine_aam_warmup"]
```

## 제공 프리셋
| 이름 | 학습 위치 | Loss | 의도 |
|---|---|---|---|
| `baseline_eval_only` | (없음) | (없음) | pretrained EER 기준 측정 |
| `adapter_pair_margin_v1` | adapter | pair_margin + distill | 기존 노트북 재현 |
| `adapter_aam` | adapter | AAM-Softmax | adapter + 원본 loss |
| `bn_affine_aam` | BN/Norm affine | AAM-Softmax | **누락 지적했던 핵심 recipe** (ECAPA에서 통한 방식 TitaNet판) |
| `bn_affine_aam_warmup` | BN affine | AAM + 2-phase | head warm-up 후 encoder 풀기 |
| `bn_affine_aam_hardneg` | BN affine | AAM + pair_margin (hard) | hard negative mining 추가 |
| `tiny_last_bn_aam` | block[4] 일부 BN | AAM-Softmax | 마지막 블록 보수적 |
| `last_block_full_aam` | block[4] 전체 | AAM-Softmax | 의도적 공격 (실패 재현 비교) |
| `pair_margin_strong_aug` | adapter | pair_margin | telephony aug 강하게 |

## 결과 저장
모든 실험은 `/content/sweep_results/{preset_name}.json`에 자동 누적.
마지막 셀에서 한 표로 비교.

## 변경된 핵심 (이전 분석 보강)
1. **AAM-Softmax loss 추가** — TitaNet 원본 loss 유지하는 옵션
2. **Head warm-up 단계 분리** — 첫 N epoch는 head만 학습 후 encoder unfreeze
3. **Hard negative mining** — batch 내 가장 가까운 다른 화자 샘플링
4. **Eval seed/n_pairs 고정** — 모든 실험 동일 조건 비교
5. **Trainable weight norm 모니터링** — adapter가 실제 학습됐는지 진단
6. **Same recipe TitaNet 비교 가능** — ECAPA `bn_affine_aam`을 TitaNet에 그대로 적용


---
## 0. 환경 셋업

In [ ]:
# NeMo + 의존성 (10분 정도 소요)
!pip install -q -U nemo_toolkit[asr]==2.7.2
!pip install -q soundfile pandas numpy scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.1/443.1 kB 16.4 MB/s eta 0:00

In [ ]:
!nvidia-smi

Sun May 10 23:52:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   44G  192G  19% /


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!python --version

Python 3.12.13


---
## 1. PRESET 정의 + CONFIG 선택

이 셀에서 `CONFIG = PRESETS["..."]`만 바꾸면 시나리오 전환됨.

**모든 프리셋이 공유하는 base config**가 있고, 프리셋은 그 위에 override만 함.

In [ ]:
# ============================================================
# Base config: 모든 프리셋의 default
# ============================================================
import copy

BASE_CONFIG = {
    # --- 식별 ---
    "preset_name": "baseline_eval_only",  # 결과 저장 파일명
    "seed": 42,

    # --- 모델 ---
    "model_name": "titanet_small",  # or "titanet_large"

    # --- Trainable recipe ---
    # "none" | "adapter_only" | "bn_affine_only" | "tiny_last_bn"
    # | "last_block_full" | "encoder_full"
    "trainable_recipe": "none",

    # --- Adapter 설정 (recipe == adapter_only일 때) ---
    "adapter_bottleneck": 32,
    "adapter_residual_scale": 0.1,

    # --- Loss type ---
    # "none" | "pair_margin" | "aam" | "triplet"
    "loss_type": "none",

    # --- Pair margin loss ---
    "lambda_same": 0.55,
    "lambda_diff": 2.0,
    "neg_margin": 0.22,

    # --- Triplet loss ---
    "triplet_margin": 0.2,

    # --- AAM-Softmax ---
    "aam_margin": 0.2,        # m, 0.1~0.4 범위
    "aam_scale": 30.0,        # s
    "aam_num_classes": None,  # None이면 train_speakers 수로 자동 설정

    # --- Distillation ---
    "use_distill": False,
    "lambda_distill": 0.7,

    # --- Hard negative mining ---
    "hard_negative": False,           # batch 내 가장 가까운 다른 화자
    "hard_negative_warmup_epochs": 1, # 첫 N epoch는 random negative

    # --- Head warm-up ---
    "head_warmup_epochs": 0,  # 첫 N epoch는 head만 학습 후 encoder unfreeze

    # --- Optimizer ---
    "optimizer": "adamw",
    "lr_adapter": 1e-4,
    "lr_bn_affine": 5e-5,
    "lr_encoder": 5e-6,
    "lr_aam_head": 1e-3,      # AAM head 신규 학습용
    "weight_decay": 1e-5,
    "grad_clip_norm": 3.0,

    # --- Scheduler ---
    "scheduler": "cosine",   # "cosine" | "constant"
    "warmup_ratio": 0.05,

    # --- Train ---
    "batch_size": 96,
    "num_workers": 4,
    "num_epochs": 3,
    "steps_per_epoch": 30000,

    # --- Augmentation ---
    "prob_clean": 1.0,
    "prob_telephony": 0.0,
    "prob_telephony_noise": 0.0,
    "spec_augment": False,    # 추가 augmentation
    "spec_freq_masks": 2,
    "spec_time_masks": 5,

    # --- Crop buckets ---
    "crop_buckets": [
        ((0.8, 1.2), 0.20),
        ((1.2, 2.0), 0.30),
        ((2.0, 4.0), 0.30),
        ((4.0, 8.0), 0.20),
    ],

    # --- Eval (모든 실험 동일 조건!) ---
    "eval_verify_durations": [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0, 5.0],
    "enrollment_sec": 3.0,
    "best_metric_verify_sec": 3.0,
    "eval_n_pairs_per_speaker": 20,
    "eval_seed": 42,           # 모든 실험에서 동일한 pair set
    "eval_apply_telephony": True,

    # --- Checkpoint guard ---
    "diff_mean_tolerance": 0.02,
    "gap_tolerance": 0.02,
}


# ============================================================
# 프리셋들: BASE_CONFIG에 override
# ============================================================
def make_preset(**overrides):
    cfg = copy.deepcopy(BASE_CONFIG)
    cfg.update(overrides)
    return cfg


PRESETS = {

    # --- 1. baseline ---
    "baseline_eval_only": make_preset(
        preset_name="baseline_eval_only",
        trainable_recipe="none",
        loss_type="none",
    ),

    # --- 2. 기존 adapter pair_margin 재현 ---
    "adapter_pair_margin_v1": make_preset(
        preset_name="adapter_pair_margin_v1",
        trainable_recipe="adapter_only",
        loss_type="pair_margin",
        use_distill=True,
        lambda_distill=0.7,
        lambda_same=0.55,
        lambda_diff=2.0,
        neg_margin=0.22,
        adapter_residual_scale=0.1,
    ),

    # --- 3. adapter + AAM ---
    "adapter_aam": make_preset(
        preset_name="adapter_aam",
        trainable_recipe="adapter_only",
        loss_type="aam",
        aam_margin=0.2,
        aam_scale=30.0,
        adapter_residual_scale=0.1,
    ),

    # --- 4. *** 핵심 누락 recipe: BN affine + AAM ***
    "bn_affine_aam": make_preset(
        preset_name="bn_affine_aam",
        trainable_recipe="bn_affine_only",
        loss_type="aam",
        aam_margin=0.2,
        aam_scale=30.0,
        lr_bn_affine=1e-5,
        lr_aam_head=1e-3,
        num_epochs=3,
        batch_size=32,                    # ← 추가
        crop_buckets=[                    # ← 추가, 긴 발화 제외
            ((0.8, 1.2), 0.25),
            ((1.2, 2.0), 0.35),
            ((2.0, 4.0), 0.30),
            ((4.0, 6.0), 0.10),
        ],
    ),

    # --- 5. BN affine + AAM + head warm-up ---
    "bn_affine_aam_warmup": make_preset(
        preset_name="bn_affine_aam_warmup",
        trainable_recipe="bn_affine_only",
        loss_type="aam",
        aam_margin=0.2,
        aam_scale=30.0,
        head_warmup_epochs=1,
        num_epochs=4,
    ),

    # --- 6. BN affine + AAM + hard negative ---
    "bn_affine_aam_hardneg": make_preset(
        preset_name="bn_affine_aam_hardneg",
        trainable_recipe="bn_affine_only",
        loss_type="aam",
        aam_margin=0.2,
        aam_scale=30.0,
        hard_negative=True,
        hard_negative_warmup_epochs=1,
        num_epochs=4,
    ),

    # --- 7. 마지막 block 일부 BN + AAM ---
    "tiny_last_bn_aam": make_preset(
        preset_name="tiny_last_bn_aam",
        trainable_recipe="tiny_last_bn",
        loss_type="aam",
        aam_margin=0.2,
        aam_scale=30.0,
        lr_bn_affine=1e-5,
        num_epochs=3,
    ),

    # --- 8. 의도적 공격 비교: 마지막 block 전체 + AAM ---
    "last_block_full_aam": make_preset(
        preset_name="last_block_full_aam",
        trainable_recipe="last_block_full",
        loss_type="aam",
        aam_margin=0.2,
        aam_scale=30.0,
        lr_encoder=1e-5,        # 보수적 LR
        head_warmup_epochs=1,   # head 먼저
        num_epochs=3,
    ),

    # --- 9. pair_margin + 강한 augmentation ---
    "pair_margin_strong_aug": make_preset(
        preset_name="pair_margin_strong_aug",
        trainable_recipe="adapter_only",
        loss_type="pair_margin",
        use_distill=True,
        lambda_distill=0.5,
        lambda_same=0.55,
        lambda_diff=2.0,
        neg_margin=0.22,
        prob_clean=0.4,
        prob_telephony=0.4,
        prob_telephony_noise=0.2,
        adapter_residual_scale=0.1,
    ),
}


# ============================================================
# *** 여기서 시나리오 선택 ***
# ============================================================
SELECTED_PRESET = "baseline_eval_only"
CONFIG = PRESETS[SELECTED_PRESET]
CONFIG["model_name"] = "titanet_large"
CONFIG["preset_name"] = "baseline_titanet_large"

print(f"선택된 프리셋: {SELECTED_PRESET} (TitaNet-Large)")
print(f"  model_name:       {CONFIG['model_name']}")
print(f"  preset_name:      {CONFIG['preset_name']}")

print(f"선택된 프리셋: {SELECTED_PRESET} (override: lr5e5, ep8)")
print(f"  trainable_recipe: {CONFIG['trainable_recipe']}")
print(f"  loss_type:        {CONFIG['loss_type']}")
print(f"  num_epochs:       {CONFIG['num_epochs']}")
print(f"  batch_size:       {CONFIG['batch_size']}")
print(f"  lr_bn_affine:     {CONFIG['lr_bn_affine']}")
print(f"  preset_name:      {CONFIG['preset_name']}")


선택된 프리셋: baseline_eval_only (TitaNet-Large)
  model_name:       titanet_large
  preset_name:      baseline_titanet_large
선택된 프리셋: baseline_eval_only (override: lr5e5, ep8)
  trainable_recipe: none
  loss_type:        none
  num_epochs:       3
  batch_size:       96
  lr_bn_affine:     5e-05
  preset_name:      baseline_titanet_large


---
## 2. 데이터 압축 해제

기존 노트북과 동일. Drive에서 sample.zip / eval.zip 가져와서 풀기.

In [ ]:
import os
from pathlib import Path

DRIVE_DATA_DIR = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/data"

WORK_DIR = "/content/work"
DATA_DIR = "/content/data"
EVAL_DIR = "/content/eval"
CHECKPOINT_DIR = "/content/ckpt_titanet_sweep"
RESULTS_DIR = "/content/sweep_results"

for d in [WORK_DIR, DATA_DIR, EVAL_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

SAMPLE_ZIP_DRIVE = f"{DRIVE_DATA_DIR}/sisicallcall_train_sample.zip"
SAMPLE_ZIP_LOCAL = f"{WORK_DIR}/sample.zip"

EVAL_ZIP_DRIVE = f"{DRIVE_DATA_DIR}/sisicallcall_eval.zip"
EVAL_ZIP_LOCAL = f"{WORK_DIR}/eval.zip"

MANIFEST_PATH = f"{DATA_DIR}/manifest.jsonl"
EVAL_INDEX_PATH = f"{EVAL_DIR}/callword_short/eval_index.csv"

print("경로 설정 완료")


경로 설정 완료


In [ ]:
import shutil
import zipfile
import time

def disk_usage_gb(path="/content"):
    s = shutil.disk_usage(path)
    return {"total_gb": round(s.total / (1024**3), 1), "free_gb": round(s.free / (1024**3), 1)}


def maybe_extract(zip_drive, zip_local, target_dir, marker_path):
    if os.path.exists(marker_path):
        print(f"이미 풀려있음: {marker_path}")
        return
    print(f"copy {zip_drive} -> {zip_local}")
    t0 = time.time()
    shutil.copy(zip_drive, zip_local)
    print(f"  copy done ({(time.time()-t0)/60:.1f}min)")

    print("extract...")
    t0 = time.time()
    with zipfile.ZipFile(zip_local, "r") as zf:
        zf.extractall(target_dir)
    print(f"  extract done ({(time.time()-t0)/60:.1f}min)")
    os.remove(zip_local)


maybe_extract(SAMPLE_ZIP_DRIVE, SAMPLE_ZIP_LOCAL, DATA_DIR, MANIFEST_PATH)
maybe_extract(EVAL_ZIP_DRIVE, EVAL_ZIP_LOCAL, EVAL_DIR, EVAL_INDEX_PATH)

print(disk_usage_gb())


copy /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/data/sisicallcall_train_sample.zip -> /content/work/sample.zip
  copy done (14.4min)
extract...
  extract done (5.4min)
copy /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/data/sisicallcall_eval.zip -> /content/work/eval.zip
  copy done (0.1min)
extract...
  extract done (0.0min)
{'total_gb': 235.7, 'free_gb': 126.1}


---
## 3. Imports + seed

In [ ]:
import os, json, random, time, audioop, math
from collections import defaultdict

import numpy as np
import pandas as pd
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_curve
from tqdm import tqdm

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


/tmp/ipykernel_12437/927196324.py:1: DeprecationWarning: 'audioop' is deprecated and slated for removal in Python 3.13
  import os, json, random, time, audioop, math


device: cuda


---
## 4. Manifest 로드 + speaker 분포

In [ ]:
assert os.path.exists(MANIFEST_PATH), f"manifest 없음: {MANIFEST_PATH}"

all_entries = []
with open(MANIFEST_PATH, encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        if not os.path.isabs(e["audio_filepath"]):
            e["audio_filepath"] = os.path.join(DATA_DIR, e["audio_filepath"])
        e["label"] = str(e["label"])
        all_entries.append(e)

by_speaker = defaultdict(list)
for e in all_entries:
    by_speaker[e["label"]].append(e)

speakers = sorted(by_speaker.keys())
speaker_to_id = {sp: i for i, sp in enumerate(speakers)}

train_speakers = [sp for sp in speakers if len(by_speaker[sp]) >= 2]

NUM_TRAIN_SPEAKERS = len(train_speakers)

# AAM head 클래스 수 자동 설정
if CONFIG["aam_num_classes"] is None:
    CONFIG["aam_num_classes"] = NUM_TRAIN_SPEAKERS

print(f"전체 발화: {len(all_entries):,}")
print(f"전체 speaker: {len(speakers):,}")
print(f"학습 가능 speaker (>=2 utt): {NUM_TRAIN_SPEAKERS:,}")
print(f"AAM head num_classes: {CONFIG['aam_num_classes']}")


전체 발화: 427,554
전체 speaker: 2,183
학습 가능 speaker (>=2 utt): 2,183
AAM head num_classes: 2183


---
## 5. 오디오 전처리 + augmentation

기존 노트북과 동일한 시시콜콜 telephony pipeline + crop 버킷.

In [ ]:
def apply_sisicallcall_pipeline(audio_16k_clean: torch.Tensor) -> torch.Tensor:
    """16k → 8k μ-law → 16k 시뮬레이션."""
    audio_16k_clean = audio_16k_clean.detach().float().cpu().clamp(-1.0, 1.0)
    audio_int16 = (audio_16k_clean * 32767.0).short().numpy().tobytes()
    audio_8k, _ = audioop.ratecv(audio_int16, 2, 1, 16000, 8000, None)
    ulaw = audioop.lin2ulaw(audio_8k, 2)
    pcm_8k = audioop.ulaw2lin(ulaw, 2)
    pcm_16k, _ = audioop.ratecv(pcm_8k, 2, 1, 8000, 16000, None)
    out = np.frombuffer(pcm_16k, dtype=np.int16).astype(np.float32) / 32768.0
    return torch.from_numpy(out).float().clamp(-1.0, 1.0)


def add_light_noise(audio: torch.Tensor, noise_scale=0.003) -> torch.Tensor:
    return (audio + torch.randn_like(audio) * noise_scale).clamp(-1.0, 1.0)


class TrainAugment:
    def __init__(self, cfg):
        self.p_clean = cfg["prob_clean"]
        self.p_tel = cfg["prob_telephony"]
        self.p_tel_noise = cfg["prob_telephony_noise"]

    def __call__(self, audio: torch.Tensor) -> torch.Tensor:
        r = random.random()
        if r < self.p_clean:
            return audio
        if r < self.p_clean + self.p_tel:
            return apply_sisicallcall_pipeline(audio)
        return add_light_noise(apply_sisicallcall_pipeline(audio))


class EvalTelephonyTransform:
    def __call__(self, audio: torch.Tensor) -> torch.Tensor:
        return apply_sisicallcall_pipeline(audio)


class VariableLengthCrop:
    def __init__(self, buckets, sr=16000):
        self.buckets = buckets
        self.sr = sr
        self.ranges = [x[0] for x in buckets]
        self.probs = [x[1] for x in buckets]
        assert abs(sum(self.probs) - 1.0) < 1e-6

    def sample_len(self):
        idx = np.random.choice(len(self.ranges), p=self.probs)
        lo, hi = self.ranges[idx]
        return int(random.uniform(lo, hi) * self.sr)

    def __call__(self, audio: torch.Tensor) -> torch.Tensor:
        if audio.ndim > 1:
            audio = audio.mean(dim=-1)
        target_len = self.sample_len()
        if len(audio) <= target_len:
            return F.pad(audio, (0, target_len - len(audio)))
        max_start = len(audio) - target_len
        r = random.random()
        if r < 0.30:
            start = random.randint(0, max(0, int(max_start * 0.30)))
        elif r < 0.70:
            center = max_start // 2
            width = max(1, int(max_start * 0.20))
            start = random.randint(max(0, center - width), min(max_start, center + width))
        else:
            start = random.randint(max(0, int(max_start * 0.70)), max_start)
        return audio[start:start + target_len]


def load_audio_entry(entry) -> torch.Tensor:
    audio, _ = sf.read(entry["audio_filepath"])
    if getattr(audio, "ndim", 1) > 1:
        audio = audio.mean(axis=1)
    audio = torch.from_numpy(audio.astype(np.float32))
    if "speech_start" in entry and "speech_end" in entry:
        s = int(float(entry["speech_start"]) * 16000)
        e = int(float(entry["speech_end"]) * 16000)
        if e > s and e <= len(audio):
            audio = audio[s:e]
    return audio.float().clamp(-1.0, 1.0)


print("오디오 전처리 OK")


오디오 전처리 OK


---
## 6. Dataset

`hard_negative=False`면 random sampling. `True`면 batch 내 cosine 가장 높은 다른 화자를 negative로 재선택 (학습 루프에서 처리).

In [ ]:
class SpeakerTripletDataset(Dataset):
    """
    anchor / positive: 같은 speaker
    negative: 다른 random speaker (hard negative는 학습 루프에서 batch 내 미니마이즈)
    """
    def __init__(self, by_speaker, train_speakers, cfg):
        self.by_speaker = by_speaker
        self.train_speakers = list(train_speakers)
        self.steps_per_epoch = cfg["steps_per_epoch"]
        self.crop = VariableLengthCrop(cfg["crop_buckets"])
        self.aug = TrainAugment(cfg)

    def __len__(self):
        return self.steps_per_epoch

    def __getitem__(self, idx):
        sp = random.choice(self.train_speakers)
        entries = self.by_speaker[sp]
        e_anchor, e_pos = random.sample(entries, 2)

        neg_sp = random.choice(self.train_speakers)
        while neg_sp == sp:
            neg_sp = random.choice(self.train_speakers)
        e_neg = random.choice(self.by_speaker[neg_sp])

        try:
            a = self.aug(self.crop(load_audio_entry(e_anchor)))
            p = self.aug(self.crop(load_audio_entry(e_pos)))
            n = self.aug(self.crop(load_audio_entry(e_neg)))
        except Exception:
            return self.__getitem__((idx + 1) % self.steps_per_epoch)

        return {
            "anchor": a, "positive": p, "negative": n,
            "speaker": speaker_to_id[sp],
            "neg_speaker": speaker_to_id[neg_sp],
        }


def pad_1d_batch(wavs):
    max_len = max(w.shape[0] for w in wavs)
    batch = torch.zeros(len(wavs), max_len, dtype=torch.float32)
    lengths = torch.zeros(len(wavs), dtype=torch.long)
    for i, w in enumerate(wavs):
        batch[i, :len(w)] = w
        lengths[i] = len(w)
    return batch, lengths


def triplet_collate_fn(batch):
    a, a_len = pad_1d_batch([b["anchor"] for b in batch])
    p, p_len = pad_1d_batch([b["positive"] for b in batch])
    n, n_len = pad_1d_batch([b["negative"] for b in batch])
    return {
        "anchor": a, "anchor_len": a_len,
        "positive": p, "positive_len": p_len,
        "negative": n, "negative_len": n_len,
        "speaker": torch.tensor([b["speaker"] for b in batch], dtype=torch.long),
        "neg_speaker": torch.tensor([b["neg_speaker"] for b in batch], dtype=torch.long),
    }


train_dataset = SpeakerTripletDataset(by_speaker, train_speakers, CONFIG)
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    collate_fn=triplet_collate_fn,
    drop_last=True,
)
print(f"steps/epoch: {len(train_loader)}")


steps/epoch: 312


---
## 7. TitaNet 로드 + embedding 추출

In [ ]:
import nemo.collections.asr as nemo_asr
import urllib.request

def load_titanet(name="titanet_small"):
    nemo_url_map = {
        "titanet_small": "https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_small/versions/1.0.1/files/titanet-s.nemo",
        "titanet_large": "https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_large/versions/1.0.0/files/titanet-l.nemo",
    }
    nemo_local = f"{WORK_DIR}/{name}.nemo"
    try:
        m = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained(model_name=name)
        print(f"NGC 로드 성공: {name}")
        return m
    except Exception as e:
        print(f"NGC 실패: {e}, .nemo 다운로드 시도")
        if not os.path.exists(nemo_local):
            urllib.request.urlretrieve(nemo_url_map[name], nemo_local)
        return nemo_asr.models.EncDecSpeakerLabelModel.restore_from(nemo_local)


model = load_titanet(CONFIG["model_name"]).to(DEVICE)
model.eval()  # 학습 중에도 eval 모드 유지 (BN running stat 보존)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


[NeMo W 2026-05-11 00:13:13 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-05-11 00:13:16 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-05-11 00:13:16 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-05-11 00:13:16 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-05-11 00:13:16 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


[NeMo I 2026-05-11 00:13:18 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_large/versions/v1/files/titanet-l.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2026-05-11 00:13:18 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-05-11 00:13:19 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-05-11 00:13:19 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method 

[NeMo I 2026-05-11 00:13:20 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.
NGC 로드 성공: titanet_large
params: 25.33M


In [ ]:
def _select_embedding_from_output(out):
    candidates = []
    if isinstance(out, torch.Tensor):
        candidates.append(out)
    elif isinstance(out, (tuple, list)):
        for x in out:
            if torch.is_tensor(x):
                candidates.append(x)
    else:
        raise RuntimeError(f"unexpected output type: {type(out)}")

    for x in candidates:
        if x.ndim == 2 and x.shape[-1] in (128, 192, 256, 512):
            return x
    for x in reversed(candidates):
        if x.ndim == 2:
            return x
    raise RuntimeError("embedding tensor not found")


def batch_extract_embedding(model, audios, lengths, grad=False):
    audios = audios.to(DEVICE).float()
    lengths = lengths.to(DEVICE).long()
    if grad:
        out = model.forward(input_signal=audios, input_signal_length=lengths)
        emb = _select_embedding_from_output(out)
    else:
        with torch.no_grad():
            out = model.forward(input_signal=audios, input_signal_length=lengths)
            emb = _select_embedding_from_output(out)
    emb = emb.float()
    return F.normalize(emb, dim=-1)


def extract_embedding_single(model, audio_tensor):
    if audio_tensor.ndim == 1:
        audio_tensor = audio_tensor.unsqueeze(0)
    lengths = torch.tensor([audio_tensor.shape[1]], dtype=torch.long)
    return batch_extract_embedding(model, audio_tensor, lengths, grad=False).squeeze(0).detach().cpu()


# Smoke test for embedding dim
batch = next(iter(train_loader))
test_emb = batch_extract_embedding(model, batch["anchor"][:2], batch["anchor_len"][:2], grad=False)
EMB_DIM = test_emb.shape[-1]
print(f"embedding dim: {EMB_DIM}")
assert EMB_DIM in (128, 192, 256), f"unexpected emb dim: {EMB_DIM}"


embedding dim: 192


---
## 8. Eval 함수 — 모든 실험 동일 seed/n_pairs 강제

이전 실험에서 baseline끼리도 수치가 달랐던 문제 보강.

In [ ]:
assert os.path.exists(EVAL_INDEX_PATH), f"eval_index.csv 없음: {EVAL_INDEX_PATH}"
eval_df = pd.read_csv(EVAL_INDEX_PATH)

eval_entries_by_speaker = defaultdict(list)
for _, row in eval_df.iterrows():
    sp = str(row["speaker_id"]).zfill(4)
    wav_path = str(row["wav_path"]).replace("\\", "/")
    filename = wav_path.split("/")[-1]
    local_path = f"{EVAL_DIR}/callword_short/{sp}/{filename}"
    eval_entries_by_speaker[sp].append({
        "audio_filepath": local_path, "label": sp,
        "duration": float(row["duration"]) if "duration" in row else None,
    })

print(f"eval speakers: {len(eval_entries_by_speaker):,}")


def load_eval_audio(entry, target_sec):
    audio, _ = sf.read(entry["audio_filepath"])
    if getattr(audio, "ndim", 1) > 1:
        audio = audio.mean(axis=1)
    audio = torch.from_numpy(audio.astype(np.float32)).float().clamp(-1.0, 1.0)
    n = int(target_sec * 16000)
    if len(audio) >= n:
        audio = audio[:n]
    else:
        audio = F.pad(audio, (0, n - len(audio)))
    return audio


def make_trial_pairs(eval_entries_by_speaker, n_pairs_per_speaker, seed):
    rng = random.Random(seed)
    speakers_eval = list(eval_entries_by_speaker.keys())
    pairs = []
    for sp in speakers_eval:
        entries = eval_entries_by_speaker[sp]
        if len(entries) < 2:
            continue
        for _ in range(n_pairs_per_speaker // 2):
            e1, e2 = rng.sample(entries, 2)
            pairs.append((e1, e2, 1))
        for _ in range(n_pairs_per_speaker // 2):
            e1 = rng.choice(entries)
            other = rng.choice(speakers_eval)
            while other == sp:
                other = rng.choice(speakers_eval)
            e2 = rng.choice(eval_entries_by_speaker[other])
            pairs.append((e1, e2, 0))
    rng.shuffle(pairs)
    return pairs


def compute_eer_details(scores, labels):
    scores = np.asarray(scores, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1.0 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    same = scores[labels == 1]
    diff = scores[labels == 0]
    return {
        "eer": float((fpr[idx] + fnr[idx]) / 2.0),
        "threshold": float(thresholds[idx]),
        "same_mean": float(same.mean()), "same_std": float(same.std()),
        "diff_mean": float(diff.mean()), "diff_std": float(diff.std()),
        "gap": float(same.mean() - diff.mean()),
        "n_same": int(len(same)), "n_diff": int(len(diff)),
    }


def evaluate_eer(model, adapter=None, verify_sec=3.0,
                 enrollment_sec=None, n_pairs_per_speaker=None,
                 apply_telephony=None, seed=None):
    """ 모든 default는 CONFIG에서 가져옴 → 모든 실험 동일 조건. """
    if enrollment_sec is None:
        enrollment_sec = CONFIG["enrollment_sec"]
    if n_pairs_per_speaker is None:
        n_pairs_per_speaker = CONFIG["eval_n_pairs_per_speaker"]
    if apply_telephony is None:
        apply_telephony = CONFIG["eval_apply_telephony"]
    if seed is None:
        seed = CONFIG["eval_seed"]

    model.eval()
    if adapter is not None:
        adapter.eval()

    eval_tf = EvalTelephonyTransform() if apply_telephony else None
    pairs = make_trial_pairs(eval_entries_by_speaker, n_pairs_per_speaker, seed)

    scores, labels = [], []
    for e1, e2, label in pairs:
        try:
            a1 = load_eval_audio(e1, enrollment_sec)
            a2 = load_eval_audio(e2, verify_sec)
            if eval_tf is not None:
                a1 = eval_tf(a1)
                a2 = eval_tf(a2)
            emb1 = extract_embedding_single(model, a1)
            emb2 = extract_embedding_single(model, a2)
            if adapter is not None:
                with torch.no_grad():
                    emb1 = adapter(emb1.unsqueeze(0).to(DEVICE)).squeeze(0).detach().cpu()
                    emb2 = adapter(emb2.unsqueeze(0).to(DEVICE)).squeeze(0).detach().cpu()
            sim = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()
            scores.append(sim)
            labels.append(label)
        except Exception:
            continue
    if len(scores) < 10:
        return None
    return compute_eer_details(scores, labels)


print("eval 함수 OK")


eval speakers: 200
eval 함수 OK


---
## 9. Baseline 평가 (pretrained 그대로)

이 baseline은 모든 프리셋에서 동일 seed로 측정되어 비교 기준이 됨.

In [ ]:
print("Pretrained TitaNet baseline 평가")
print(f"  eval seed:      {CONFIG['eval_seed']}")
print(f"  n_pairs/spk:    {CONFIG['eval_n_pairs_per_speaker']}")
print(f"  telephony:      {CONFIG['eval_apply_telephony']}")
print()

baseline_results = {}
for sec in CONFIG["eval_verify_durations"]:
    res = evaluate_eer(model, adapter=None, verify_sec=sec)
    baseline_results[sec] = res
    if res:
        print(f"EER@{sec:.1f}s: {res['eer']*100:.2f}% | "
              f"thr={res['threshold']:.3f} | "
              f"same={res['same_mean']:.3f} | "
              f"diff={res['diff_mean']:.3f} | "
              f"gap={res['gap']:.3f}")

baseline_3s = baseline_results[CONFIG["best_metric_verify_sec"]]
BASELINE_EER_3S = baseline_3s["eer"]
BASELINE_DIFF_3S = baseline_3s["diff_mean"]
BASELINE_GAP_3S = baseline_3s["gap"]

print()
print(f"BASELINE @ {CONFIG['best_metric_verify_sec']}s")
print(f"  EER  = {BASELINE_EER_3S*100:.2f}%")
print(f"  diff = {BASELINE_DIFF_3S:.3f}")
print(f"  gap  = {BASELINE_GAP_3S:.3f}")


Pretrained TitaNet baseline 평가
  eval seed:      42
  n_pairs/spk:    20
  telephony:      True

EER@0.5s: 42.02% | thr=0.110 | same=0.137 | diff=0.097 | gap=0.040
EER@0.8s: 33.25% | thr=0.145 | same=0.209 | diff=0.110 | gap=0.099
EER@1.0s: 29.83% | thr=0.163 | same=0.236 | diff=0.116 | gap=0.119
EER@1.2s: 28.45% | thr=0.178 | same=0.260 | diff=0.126 | gap=0.134
EER@1.5s: 24.75% | thr=0.196 | same=0.291 | diff=0.135 | gap=0.156
EER@2.0s: 22.50% | thr=0.228 | same=0.335 | diff=0.155 | gap=0.180
EER@3.0s: 20.65% | thr=0.270 | same=0.382 | diff=0.179 | gap=0.202
EER@5.0s: 23.97% | thr=0.266 | same=0.378 | diff=0.191 | gap=0.187

BASELINE @ 3.0s
  EER  = 20.65%
  diff = 0.179
  gap  = 0.202


---
## 10. (baseline_eval_only면 여기서 결과 저장 후 종료)

In [ ]:
def save_results(payload):
    path = f"{RESULTS_DIR}/{CONFIG['preset_name']}.json"
    with open(path, "w") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print(f"saved: {path}")


if CONFIG["preset_name"] == "baseline_eval_only" or CONFIG["loss_type"] == "none":
    payload = {
        "preset_name": CONFIG["preset_name"],
        "config": {k: v for k, v in CONFIG.items() if k != "crop_buckets"},
        "baseline_results": {f"{k}s": v for k, v in baseline_results.items()},
        "final_results": {f"{k}s": v for k, v in baseline_results.items()},
        "history": [],
        "note": "baseline_eval_only — 학습 없음",
    }
    save_results(payload)
    print("\n>>> baseline_eval_only: 다음 셀들은 실행 안 해도 됨. 다른 PRESET으로 바꿔서 다시 실행.")


saved: /content/sweep_results/baseline_titanet_large.json

>>> baseline_eval_only: 다음 셀들은 실행 안 해도 됨. 다른 PRESET으로 바꿔서 다시 실행.


---
## 11. Adapter, AAM Head 정의

In [ ]:
class EmbeddingAdapter(nn.Module):
    def __init__(self, emb_dim=192, bottleneck=32, residual_scale=0.1):
        super().__init__()
        self.residual_scale = residual_scale
        self.net = nn.Sequential(
            nn.Linear(emb_dim, bottleneck),
            nn.ReLU(),
            nn.Linear(bottleneck, emb_dim),
        )

    def forward(self, emb):
        out = emb + self.residual_scale * self.net(emb)
        return F.normalize(out, dim=-1)


class AAMSoftmaxHead(nn.Module):
    """
    Additive Angular Margin Softmax.
    cos(θ + m)을 target class에 적용 후 scale s로 logits 만듦.
    cross_entropy(logits, label) 그대로 쓰면 됨.
    """
    def __init__(self, emb_dim, num_classes, margin=0.2, scale=30.0):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        # cos(π - m) threshold for numerical stability
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin
        self.weight = nn.Parameter(torch.randn(num_classes, emb_dim) * 0.01)

    def forward(self, emb, labels=None):
        # emb is already L2-normalized
        W = F.normalize(self.weight, dim=-1)
        cosine = F.linear(emb, W).clamp(-1.0 + 1e-7, 1.0 - 1e-7)

        if labels is None:
            # inference logits
            return cosine * self.scale

        sine = torch.sqrt(1.0 - cosine.pow(2))
        phi = cosine * self.cos_m - sine * self.sin_m
        # for cosine < th, use cosine - mm (avoid wrap-around)
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        logits = (one_hot * phi + (1.0 - one_hot) * cosine) * self.scale
        return logits


# 인스턴스 생성
adapter = None
aam_head = None

if CONFIG["trainable_recipe"] == "adapter_only":
    adapter = EmbeddingAdapter(
        emb_dim=EMB_DIM,
        bottleneck=CONFIG["adapter_bottleneck"],
        residual_scale=CONFIG["adapter_residual_scale"],
    ).to(DEVICE)
    print(f"Adapter: {sum(p.numel() for p in adapter.parameters()):,} params")

if CONFIG["loss_type"] == "aam":
    # 안전장치: CONFIG이 다시 빌드되면 aam_num_classes가 None으로 돌아감
    if CONFIG["aam_num_classes"] is None:
        CONFIG["aam_num_classes"] = NUM_TRAIN_SPEAKERS
        print(f"  (auto-set) aam_num_classes = {NUM_TRAIN_SPEAKERS}")

    aam_head = AAMSoftmaxHead(
        emb_dim=EMB_DIM,
        num_classes=CONFIG["aam_num_classes"],
        margin=CONFIG["aam_margin"],
        scale=CONFIG["aam_scale"],
    ).to(DEVICE)
    print(f"AAM head: {sum(p.numel() for p in aam_head.parameters()):,} params, "
          f"m={CONFIG['aam_margin']}, s={CONFIG['aam_scale']}, "
          f"num_classes={CONFIG['aam_num_classes']}")


---
## 12. Trainable parameter recipe

이전 분석에서 짚은 핵심:
- `bn_affine_only` ← ECAPA에서 통한 recipe TitaNet 그대로 적용
- `last_block_full` ← block[4]가 사실상 encoder 대부분이므로 위험. 비교용으로만.

In [ ]:
def freeze_all(m):
    for p in m.parameters():
        p.requires_grad = False


def unfreeze_bn_affine(model):
    """BN/Norm 계열 weight/bias만 학습."""
    freeze_all(model)
    trainable = []
    for module_name, module in model.named_modules():
        cls = module.__class__.__name__.lower()
        if not ("batchnorm" in cls or "batch_norm" in cls or "norm" in cls or ".bn" in module_name.lower()):
            continue
        for pname, p in module.named_parameters(recurse=False):
            if pname in ("weight", "bias"):
                p.requires_grad = True
                trainable.append(p)
    return trainable


def unfreeze_tiny_last_bn(model):
    """encoder.encoder.4 내부의 norm/affine만, conv.weight 제외."""
    freeze_all(model)
    trainable = []
    for name, p in model.named_parameters():
        ok = (
            name.startswith("encoder.encoder.4")
            and (name.endswith(".weight") or name.endswith(".bias"))
            and (".mconv.2." in name or ".mconv.7." in name or ".mconv.12." in name or ".res." in name)
        )
        if "conv.weight" in name: ok = False
        if ".fc." in name: ok = False
        if ok:
            p.requires_grad = True
            trainable.append(p)
    return trainable


def unfreeze_last_block_full(model):
    """encoder.encoder.4 전체 unfreeze (의도적 공격, 비교용)."""
    freeze_all(model)
    trainable = []
    for name, p in model.named_parameters():
        if name.startswith("encoder.encoder.4"):
            p.requires_grad = True
            trainable.append(p)
    return trainable


def unfreeze_encoder_full(model):
    trainable = []
    for p in model.parameters():
        p.requires_grad = True
        trainable.append(p)
    return trainable


# Recipe 적용
freeze_all(model)
encoder_trainable = []

recipe = CONFIG["trainable_recipe"]
if recipe == "none":
    pass
elif recipe == "adapter_only":
    pass  # encoder는 freeze, adapter만 학습
elif recipe == "bn_affine_only":
    encoder_trainable = unfreeze_bn_affine(model)
elif recipe == "tiny_last_bn":
    encoder_trainable = unfreeze_tiny_last_bn(model)
elif recipe == "last_block_full":
    encoder_trainable = unfreeze_last_block_full(model)
elif recipe == "encoder_full":
    encoder_trainable = unfreeze_encoder_full(model)
else:
    raise ValueError(f"unknown recipe: {recipe}")

n_encoder_trainable = sum(p.numel() for p in encoder_trainable)
n_adapter = sum(p.numel() for p in adapter.parameters()) if adapter else 0
n_aam = sum(p.numel() for p in aam_head.parameters()) if aam_head else 0

print(f"recipe: {recipe}")
print(f"  encoder trainable: {n_encoder_trainable:,}")
print(f"  adapter:           {n_adapter:,}")
print(f"  aam head:          {n_aam:,}")
print(f"  total trainable:   {n_encoder_trainable + n_adapter + n_aam:,}")


recipe: none
  encoder trainable: 0
  adapter:           0
  aam head:          0
  total trainable:   0


---
## 13. Optimizer + scheduler + head warm-up 지원

`head_warmup_epochs > 0`이면 첫 N epoch는 encoder freeze 상태로 head/adapter만 학습.

In [ ]:
def build_optimizer(use_encoder=True):
    """
    head_warmup phase에서는 use_encoder=False로 빌드.
    """
    param_groups = []

    if adapter is not None:
        param_groups.append({
            "params": list(adapter.parameters()),
            "lr": CONFIG["lr_adapter"],
            "name": "adapter",
        })

    if aam_head is not None:
        param_groups.append({
            "params": list(aam_head.parameters()),
            "lr": CONFIG["lr_aam_head"],
            "name": "aam_head",
        })

    if use_encoder and len(encoder_trainable) > 0:
        if recipe in ("bn_affine_only", "tiny_last_bn"):
            lr_enc = CONFIG["lr_bn_affine"]
        else:
            lr_enc = CONFIG["lr_encoder"]
        param_groups.append({
            "params": encoder_trainable,
            "lr": lr_enc,
            "name": "encoder",
        })

    if not param_groups:
        return None

    opt = torch.optim.AdamW(param_groups, weight_decay=CONFIG["weight_decay"])
    return opt


def build_scheduler(optimizer, total_steps):
    if optimizer is None or CONFIG["scheduler"] == "constant":
        return None
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


optimizer = build_optimizer(use_encoder=(CONFIG["head_warmup_epochs"] == 0))
total_steps = CONFIG["num_epochs"] * CONFIG["steps_per_epoch"] // CONFIG["batch_size"]
scheduler = build_scheduler(optimizer, total_steps)

if optimizer:
    print("optimizer param groups:")
    for g in optimizer.param_groups:
        print(f"  {g.get('name', '?'):10s} lr={g['lr']:.2e} n={sum(p.numel() for p in g['params']):,}")
else:
    print("optimizer 없음")


optimizer 없음


---
## 14. Loss 함수들 (pair_margin / aam / triplet)

In [ ]:
def cosine_distill_loss(student_emb, teacher_emb):
    student_emb = F.normalize(student_emb, dim=-1)
    teacher_emb = F.normalize(teacher_emb, dim=-1)
    return (1.0 - F.cosine_similarity(student_emb, teacher_emb, dim=-1)).mean()


def pair_margin_losses(a, p, n, neg_margin):
    a = F.normalize(a, dim=-1)
    p = F.normalize(p, dim=-1)
    n = F.normalize(n, dim=-1)
    same_sim = F.cosine_similarity(a, p, dim=-1)
    diff_sim = F.cosine_similarity(a, n, dim=-1)
    same_loss = (1.0 - same_sim).mean()
    diff_loss = F.relu(diff_sim - neg_margin).mean()
    return same_loss, diff_loss, same_sim.mean().item(), diff_sim.mean().item()


def triplet_loss(a, p, n, margin):
    a = F.normalize(a, dim=-1)
    p = F.normalize(p, dim=-1)
    n = F.normalize(n, dim=-1)
    pos_sim = F.cosine_similarity(a, p, dim=-1)
    neg_sim = F.cosine_similarity(a, n, dim=-1)
    return F.relu(neg_sim - pos_sim + margin).mean(), pos_sim.mean().item(), neg_sim.mean().item()


def hard_negative_resample(a_emb, neg_embs, neg_speakers, anchor_speakers):
    """
    배치 내에서, anchor와 다른 speaker 중 cosine 가장 높은 것을 negative로 재선택.
    a_emb, neg_embs: (B, D), normalized
    """
    with torch.no_grad():
        sim = a_emb @ neg_embs.t()  # (B, B)
        # anchor speaker == neg_speaker인 곳은 -inf로 마스크 (실제로는 다른 speaker만 들어있지만 안전)
        mask = (anchor_speakers.unsqueeze(1) == neg_speakers.unsqueeze(0))
        sim = sim.masked_fill(mask, -1e4)
        hard_idx = sim.argmax(dim=-1)
    return hard_idx


print("loss 함수 OK")


loss 함수 OK


---
## 15. (Distillation 사용 시) Teacher 모델 로드

In [ ]:
teacher_model = None
if CONFIG["use_distill"]:
    teacher_model = load_titanet(CONFIG["model_name"]).to(DEVICE)
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad = False
    print("teacher loaded & frozen")
else:
    print("distill 미사용")


distill 미사용


---
## 16. 학습 루프

핵심:
- `model.train()` 절대 호출 안 함 (BN running stat 변경 방지)
- adapter/aam_head는 `.train()` 호출 OK
- head_warmup_epochs 동안은 encoder freeze (use_encoder=False optimizer)
- hard_negative_warmup_epochs 동안은 random negative

매 epoch마다:
- trainable param의 weight norm 변화량 출력 (학습됐는지 진단)
- EER@3.0/1.2/0.8s 측정 + checkpoint guard 통과 시 저장

In [ ]:
best_eer = BASELINE_EER_3S
best_ckpt_path = f"{CHECKPOINT_DIR}/best_{CONFIG['preset_name']}.pt"
history = []

def is_acceptable(res_3s):
    if res_3s is None:
        return False
    eer_ok = res_3s["eer"] < BASELINE_EER_3S
    diff_ok = res_3s["diff_mean"] <= BASELINE_DIFF_3S + CONFIG["diff_mean_tolerance"]
    gap_ok = res_3s["gap"] >= BASELINE_GAP_3S - CONFIG["gap_tolerance"]
    return eer_ok and diff_ok and gap_ok


def trainable_norm_snapshot():
    """학습됐는지 검증용 weight norm 스냅샷."""
    snap = {}
    if adapter is not None:
        snap["adapter"] = sum(p.detach().norm().item() for p in adapter.parameters())
    if aam_head is not None:
        snap["aam_head"] = aam_head.weight.detach().norm().item()
    if len(encoder_trainable) > 0:
        snap["encoder"] = sum(p.detach().norm().item() for p in encoder_trainable)
    return snap


initial_norms = trainable_norm_snapshot()
print("initial weight norms:", initial_norms)
print()


# Train modes
def set_train_modules_train():
    if adapter is not None:
        adapter.train()
    if aam_head is not None:
        aam_head.train()


def set_train_modules_eval():
    if adapter is not None:
        adapter.eval()
    if aam_head is not None:
        aam_head.eval()


if CONFIG["loss_type"] == "none":
    print("loss_type=none → 학습 생략")
else:
    for epoch in range(CONFIG["num_epochs"]):
        # === phase 결정 ===
        in_warmup = epoch < CONFIG["head_warmup_epochs"]
        use_hard_neg = (
            CONFIG["hard_negative"]
            and epoch >= CONFIG["hard_negative_warmup_epochs"]
        )

        # head_warmup이 끝나는 epoch에서 optimizer 재구성 (encoder unfreeze)
        if epoch == CONFIG["head_warmup_epochs"] and CONFIG["head_warmup_epochs"] > 0:
            print(">>> head warmup 종료, encoder 풀고 optimizer 재구성")
            optimizer = build_optimizer(use_encoder=True)
            scheduler = build_scheduler(optimizer, total_steps)

        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} "
              f"| warmup={in_warmup} | hard_neg={use_hard_neg}")
        print(f"{'='*60}")

        model.eval()
        if teacher_model is not None:
            teacher_model.eval()
        set_train_modules_train()

        losses, dist_losses, ce_losses, same_losses, diff_losses = [], [], [], [], []
        same_sims, diff_sims = [], []

        pbar = tqdm(train_loader, desc=f"epoch {epoch+1}")
        for batch in pbar:
            a = batch["anchor"].to(DEVICE)
            al = batch["anchor_len"].to(DEVICE)
            p = batch["positive"].to(DEVICE)
            pl = batch["positive_len"].to(DEVICE)
            n = batch["negative"].to(DEVICE)
            nl = batch["negative_len"].to(DEVICE)

            speaker = batch["speaker"].to(DEVICE)
            neg_speaker = batch["neg_speaker"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            # encoder gradient는 adapter recipe가 아닐 때만 흐름
            grad_through_encoder = (recipe != "adapter_only" and not in_warmup)

            a_emb = batch_extract_embedding(model, a, al, grad=grad_through_encoder)
            p_emb = batch_extract_embedding(model, p, pl, grad=grad_through_encoder)
            n_emb = batch_extract_embedding(model, n, nl, grad=grad_through_encoder)

            # adapter 적용
            if adapter is not None:
                # encoder gradient가 흐르면 detach 안 해도 되지만, 안전하게 detach
                if grad_through_encoder:
                    a_emb = adapter(a_emb)
                    p_emb = adapter(p_emb)
                    n_emb = adapter(n_emb)
                else:
                    a_emb = adapter(a_emb.detach())
                    p_emb = adapter(p_emb.detach())
                    n_emb = adapter(n_emb.detach())

            # hard negative resample
            if use_hard_neg:
                with torch.no_grad():
                    a_emb_norm = F.normalize(a_emb, dim=-1)
                    n_emb_norm = F.normalize(n_emb, dim=-1)
                    hard_idx = hard_negative_resample(
                        a_emb_norm, n_emb_norm, neg_speaker, speaker,
                    )
                n_emb = n_emb[hard_idx]
                neg_speaker = neg_speaker[hard_idx]

            # === Loss ===
            total_loss = 0.0

            # distill (pair/triplet과 함께 사용)
            dist_loss_v = 0.0
            if CONFIG["use_distill"]:
                with torch.no_grad():
                    ta = batch_extract_embedding(teacher_model, a, al, grad=False)
                    tp = batch_extract_embedding(teacher_model, p, pl, grad=False)
                    tn_full = batch_extract_embedding(teacher_model, n, nl, grad=False)
                if use_hard_neg:
                    tn = tn_full[hard_idx]
                else:
                    tn = tn_full
                dist_loss = (
                    cosine_distill_loss(a_emb, ta)
                    + cosine_distill_loss(p_emb, tp)
                    + cosine_distill_loss(n_emb, tn)
                ) / 3.0
                total_loss = total_loss + CONFIG["lambda_distill"] * dist_loss
                dist_loss_v = dist_loss.item()

            # main loss
            same_loss_v = diff_loss_v = ce_loss_v = 0.0
            same_sim_v = diff_sim_v = 0.0

            if CONFIG["loss_type"] == "pair_margin":
                same_loss, diff_loss, same_sim_v, diff_sim_v = pair_margin_losses(
                    a_emb, p_emb, n_emb, neg_margin=CONFIG["neg_margin"],
                )
                total_loss = (
                    total_loss
                    + CONFIG["lambda_same"] * same_loss
                    + CONFIG["lambda_diff"] * diff_loss
                )
                same_loss_v = same_loss.item()
                diff_loss_v = diff_loss.item()

            elif CONFIG["loss_type"] == "triplet":
                tri, same_sim_v, diff_sim_v = triplet_loss(
                    a_emb, p_emb, n_emb, margin=CONFIG["triplet_margin"],
                )
                total_loss = total_loss + tri
                same_loss_v = tri.item()

            elif CONFIG["loss_type"] == "aam":
                # AAM은 anchor/positive/negative 각각의 embedding을 그 화자 label로 분류
                logits_a = aam_head(F.normalize(a_emb, dim=-1), speaker)
                logits_p = aam_head(F.normalize(p_emb, dim=-1), speaker)
                logits_n = aam_head(F.normalize(n_emb, dim=-1), neg_speaker)

                ce = (
                    F.cross_entropy(logits_a, speaker)
                    + F.cross_entropy(logits_p, speaker)
                    + F.cross_entropy(logits_n, neg_speaker)
                ) / 3.0
                total_loss = total_loss + ce
                ce_loss_v = ce.item()

                # 진단용으로 same/diff sim도 측정
                with torch.no_grad():
                    same_sim_v = F.cosine_similarity(
                        F.normalize(a_emb, dim=-1),
                        F.normalize(p_emb, dim=-1),
                        dim=-1,
                    ).mean().item()
                    diff_sim_v = F.cosine_similarity(
                        F.normalize(a_emb, dim=-1),
                        F.normalize(n_emb, dim=-1),
                        dim=-1,
                    ).mean().item()

            else:
                raise ValueError(f"unknown loss_type: {CONFIG['loss_type']}")

            total_loss.backward()

            all_trainable = []
            if adapter is not None:
                all_trainable.extend(adapter.parameters())
            if aam_head is not None:
                all_trainable.extend(aam_head.parameters())
            if not in_warmup:
                all_trainable.extend(encoder_trainable)
            torch.nn.utils.clip_grad_norm_(all_trainable, CONFIG["grad_clip_norm"])

            optimizer.step()
            if scheduler is not None:
                scheduler.step()

            losses.append(total_loss.item())
            dist_losses.append(dist_loss_v)
            ce_losses.append(ce_loss_v)
            same_losses.append(same_loss_v)
            diff_losses.append(diff_loss_v)
            same_sims.append(same_sim_v)
            diff_sims.append(diff_sim_v)

            pbar.set_postfix({
                "loss": f"{np.mean(losses):.3f}",
                "same": f"{np.mean(same_sims):.3f}",
                "diff": f"{np.mean(diff_sims):.3f}",
            })

        # === epoch 끝, weight norm + EER ===
        cur_norms = trainable_norm_snapshot()
        norm_delta = {k: cur_norms[k] - initial_norms[k] for k in cur_norms}
        print(f"weight norm Δ vs init: {norm_delta}")

        set_train_modules_eval()
        res_3s = evaluate_eer(model, adapter, verify_sec=3.0)
        res_12s = evaluate_eer(model, adapter, verify_sec=1.2)
        res_08s = evaluate_eer(model, adapter, verify_sec=0.8)

        for name, res in [("3.0s", res_3s), ("1.2s", res_12s), ("0.8s", res_08s)]:
            if res:
                print(f"  EER@{name}: {res['eer']*100:.2f}% | "
                      f"thr={res['threshold']:.3f} | "
                      f"same={res['same_mean']:.3f} | "
                      f"diff={res['diff_mean']:.3f} | "
                      f"gap={res['gap']:.3f}")

        history.append({
            "epoch": epoch + 1,
            "loss": float(np.mean(losses)),
            "dist_loss": float(np.mean(dist_losses)),
            "ce_loss": float(np.mean(ce_losses)),
            "same_loss": float(np.mean(same_losses)),
            "diff_loss": float(np.mean(diff_losses)),
            "train_same_sim": float(np.mean(same_sims)),
            "train_diff_sim": float(np.mean(diff_sims)),
            "weight_norm_delta": norm_delta,
            "eer_3s": res_3s["eer"] if res_3s else None,
            "eer_12s": res_12s["eer"] if res_12s else None,
            "eer_08s": res_08s["eer"] if res_08s else None,
            "diff_mean_3s": res_3s["diff_mean"] if res_3s else None,
            "gap_3s": res_3s["gap"] if res_3s else None,
        })

        if is_acceptable(res_3s):
            best_eer = res_3s["eer"]
            payload = {
                "preset_name": CONFIG["preset_name"],
                "config": {k: v for k, v in CONFIG.items() if k != "crop_buckets"},
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "adapter_state_dict": adapter.state_dict() if adapter else None,
                "aam_head_state_dict": aam_head.state_dict() if aam_head else None,
                "best_eer_3s": best_eer,
                "baseline_eer_3s": BASELINE_EER_3S,
            }
            torch.save(payload, best_ckpt_path)
            print(f"  ✓ saved: {best_ckpt_path}")
        else:
            if res_3s:
                print(f"  not saved | "
                      f"eer={res_3s['eer']*100:.2f}% (base={BASELINE_EER_3S*100:.2f}%) | "
                      f"diff={res_3s['diff_mean']:.3f} (base={BASELINE_DIFF_3S:.3f}) | "
                      f"gap={res_3s['gap']:.3f} (base={BASELINE_GAP_3S:.3f})")

print("\n학습 완료")


initial weight norms: {}

loss_type=none → 학습 생략

학습 완료


---
## 17. 최종 평가 + JSON 결과 저장

In [ ]:
if CONFIG["loss_type"] != "none":
    if os.path.exists(best_ckpt_path):
        print(f"best ckpt 로드: {best_ckpt_path}")
        ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        if adapter and ckpt.get("adapter_state_dict"):
            adapter.load_state_dict(ckpt["adapter_state_dict"])
        if aam_head and ckpt.get("aam_head_state_dict"):
            aam_head.load_state_dict(ckpt["aam_head_state_dict"])
    else:
        print("best ckpt 없음 (baseline 못 넘었음). 마지막 epoch 모델로 평가")

    final_results = {}
    for sec in CONFIG["eval_verify_durations"]:
        res = evaluate_eer(model, adapter, verify_sec=sec)
        final_results[sec] = res
        if res:
            print(f"EER@{sec:.1f}s: {res['eer']*100:.2f}% | "
                  f"thr={res['threshold']:.3f} | "
                  f"same={res['same_mean']:.3f}±{res['same_std']:.3f} | "
                  f"diff={res['diff_mean']:.3f}±{res['diff_std']:.3f} | "
                  f"gap={res['gap']:.3f}")
else:
    final_results = baseline_results

# JSON 저장
payload = {
    "preset_name": CONFIG["preset_name"],
    "config": {k: v for k, v in CONFIG.items() if k != "crop_buckets"},
    "baseline_results": {f"{k}s": v for k, v in baseline_results.items()},
    "final_results": {f"{k}s": v for k, v in final_results.items()},
    "history": history,
    "best_eer_3s": best_eer if CONFIG["loss_type"] != "none" else BASELINE_EER_3S,
    "baseline_eer_3s": BASELINE_EER_3S,
    "improvement_pp": (BASELINE_EER_3S - best_eer) * 100 if CONFIG["loss_type"] != "none" else 0.0,
}

save_results(payload)


saved: /content/sweep_results/baseline_titanet_large.json


In [ ]:
# ============================================================
# Drive에 checkpoint + 결과 백업
# ============================================================
import shutil
from datetime import datetime

# Drive 백업 폴더
DRIVE_BACKUP_DIR = (
    "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/"
    "오브콜스(Of-Calls)/화자검증 데이터/titanet_sweep_runs"
)
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 타임스탬프 폴더
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = f"{DRIVE_BACKUP_DIR}/{ts}"
os.makedirs(run_dir, exist_ok=True)

# 1) .pt 체크포인트 전부 복사
ckpt_files = sorted(os.listdir(CHECKPOINT_DIR))
print(f"체크포인트 {len(ckpt_files)}개 백업:")
for f in ckpt_files:
    src = f"{CHECKPOINT_DIR}/{f}"
    dst = f"{run_dir}/{f}"
    shutil.copy(src, dst)
    size_mb = os.path.getsize(src) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

# 2) JSON 결과도 같이 백업
result_files = sorted(os.listdir(RESULTS_DIR))
print(f"\n결과 JSON {len(result_files)}개 백업:")
for f in result_files:
    shutil.copy(f"{RESULTS_DIR}/{f}", f"{run_dir}/{f}")
    print(f"  {f}")

print(f"\n백업 완료: {run_dir}")

체크포인트 0개 백업:

결과 JSON 1개 백업:
  baseline_titanet_large.json

백업 완료: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/오브콜스(Of-Calls)/화자검증 데이터/titanet_sweep_runs/20260511_002703


---
## 18. 누적된 모든 실험 비교 표

여러 PRESET을 한 번씩 다 돌린 후 이 셀을 실행하면, `/content/sweep_results/`에 저장된 모든 실험 결과를 한 표로 정리해줌.

In [ ]:
import glob

result_files = sorted(glob.glob(f"{RESULTS_DIR}/*.json"))
print(f"누적 실험 수: {len(result_files)}")

rows = []
for path in result_files:
    with open(path) as f:
        r = json.load(f)
    final = r["final_results"]
    base = r["baseline_results"]
    final_3s = final.get("3.0s") or {}
    base_3s = base.get("3.0s") or {}
    rows.append({
        "preset": r["preset_name"],
        "EER@3s": f"{final_3s.get('eer', 0)*100:.2f}%" if final_3s else "—",
        "EER@1.2s": f"{final.get('1.2s', {}).get('eer', 0)*100:.2f}%" if final.get("1.2s") else "—",
        "EER@0.8s": f"{final.get('0.8s', {}).get('eer', 0)*100:.2f}%" if final.get("0.8s") else "—",
        "same@3s": f"{final_3s.get('same_mean', 0):.3f}" if final_3s else "—",
        "diff@3s": f"{final_3s.get('diff_mean', 0):.3f}" if final_3s else "—",
        "gap@3s": f"{final_3s.get('gap', 0):.3f}" if final_3s else "—",
        "Δ EER vs base": f"{(base_3s.get('eer', 0) - final_3s.get('eer', 0))*100:+.2f}pp" if final_3s and base_3s else "—",
        "loss": r["config"]["loss_type"],
        "recipe": r["config"]["trainable_recipe"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# 마크다운 저장
md_path = f"{RESULTS_DIR}/_comparison.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Sweep 결과 비교\n\n")
    f.write(df.to_markdown(index=False))
print(f"\nsaved: {md_path}")


---
## 19. 사용 가이드 + "이 정도 했는데도 안 되면"

### 권장 실행 순서
1. `baseline_eval_only` — 기준 EER 측정
2. `bn_affine_aam` — 가장 가능성 높은 recipe (이전 분석에서 누락 지적)
3. `bn_affine_aam_warmup` — head warm-up이 차이를 만드는지
4. `bn_affine_aam_hardneg` — hard negative가 차이를 만드는지
5. `tiny_last_bn_aam` — 더 보수적 unfreeze
6. `adapter_aam` — adapter도 AAM과 함께 시도
7. `last_block_full_aam` — block[4] 전체 풀어도 AAM/warm-up이면 살아나는지
8. `pair_margin_strong_aug` — telephony augmentation 강하게
9. `adapter_pair_margin_v1` — 기존 실험 재현 (sanity check)

### 진단 신호
- **weight norm Δ ≈ 0**: optimizer/lr 문제로 학습 자체가 안 일어났음
- **same↑ diff↑ 동반 상승**: 모델이 모든 임베딩을 한 점으로 모음 (collapse)
- **EER 변화 < 0.5pp**: noise 수준. 더 많은 데이터가 필요한 신호

### "이 정도 해도 baseline 못 넘으면" 의미하는 것
- AAM 원본 loss 유지 → 학습 신호 정렬됨에도 안 되면 **데이터 규모 한계**
- BN affine만으로 안전하게 → 그래도 안 되면 **현재 화자 수가 TitaNet pretrained을 흔들기에 절대 부족**
- hard negative + warm-up까지 → 가능한 안전장치 다 동원한 것
- → 메모리에 기록된 결정 그대로: **실서비스 데이터 누적 후 재시도**가 합리적
